# Frozen held-out C/D evaluation: 100 cases, ten sites
1. Finish Drive sync. 2. Select L4 or A100 if available (T4 also supported). 3. Run all four code cells. 4. Send the final Latest report.
This is the final held-out run for the frozen pipeline, not another tuning pilot. It uses 100 cases selected without fault labels or model scores, ten per held-out site. No training or validation site appears in this sample. Class frequencies are not forced to balance.
Up to 400 Qwen calls: 100 classifications and up to 100 explanations per condition. This is a substantially longer run. Every attempted response is saved to Drive. If interrupted, reconnect and rerun all cells to resume pending work; cached failures are not silently retried. Do not modify the model, prompts, corpus, token limits or sample after inspecting results.
No LLM judge runs. Final scoring reports macro-F1, accuracy, per-class metrics, separate completion measures, and paired site-cluster confidence intervals. Semantic explanation quality remains a stated limitation.
Test labels are stored separately and are opened by the inference runner only after all required attempts have finished. Pending requests or invalid cache provenance stop scoring.


In [1]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Select Runtime > Disconnect and delete runtime, reconnect and rerun this cell. Complete Google authorisation. No diagnosis has started.') from exc
if not (ROOT / 'Files/phm_heldout_export.py').is_file():
    raise FileNotFoundError(f'Drive is connected but the updated project is missing at {ROOT}. Finish sync or locate CP5403 in the Files panel. Do not force-remount.')
os.chdir(ROOT)
EXPECTED = {'phm_diagnosis.py': 'a0b2af61d7d1a1c16512359f1e8029aae48380fc4b698c188f1f8caacf14b2fd', 'phm_diagnosis_qwen.py': 'f3c9938f7fd2638de842139db5f5d2716c57f4839860291279e45c4d47e049bb', 'phm_output_schema.py': '01913f0c804300723c841d7c689be5105749c1e3d9c0bc110d7a9b204ea048e9', 'phm_condition_d.py': 'e22457642e68013b1cfd4201bb6951630711d7d4890fdb28659d70dafa7a80c6', 'phm_evidence_selection.py': 'bb8cd412496f3d055d11b6d0f4ac47e970f24d43b4c19493d9ff57fca8b4fac9', 'phm_filtered_selection_pilot.py': '846c072def4f68fed21e04f366e5fea9c26da7f7547e238dc99bcbce3f42642e', 'phm_two_stage.py': '07791b9b7bedc092292edfbbd34ec4ade1c1b983809860e2c8102d8672261d6c', 'phm_literature_index.py': '725709141868ae494f1c515f67df5aec4451c0d137d461b3991c7ba436fb7f37', 'phm_run_reporting.py': 'fa54efc1c88773ca86e0e1a2343ff83dc34fff5a3d2545dd0a8c9e51415da2a2', 'phm_heldout_export.py': '7589a9bcfdf62fc8b79cd5770e01f2b2c62bc69b0f82936756e3a43d3a7ca93d', 'experiment_config.json': '563acb189f79b8f6335900a164502d01895409bee9747ee6a0f70073c63e9b3b', 'literature_retrieval_policy_v1.json': '728102ca47c5074f43f7c2ea37a19bbcf0d9a91d896dde530ccf77b8d522ab33', 'phm_final_metrics.py': '2ab1bba259b2bd7b4b3572d26bf0a78cb388bacdb64c548898fac4df7474c139', 'test_phm_heldout.py': '07890aa9038e005f5a8151ff5fd8b23fb2a0460e39103d0f1e969ed143744fc1', 'test_phm_two_stage.py': '9b92bde11d965817c16c9459e2b9cf89b6a9ad93467bd2216a4eade9e987d829', 'requirements-literature-colab.txt': '9dbbad4b5ea0136c7c6b0b4a0b0f309a62ced876b9ed9a56f662c5d593d7755c'}
EXPECTED_CONTRACT = '58e29fccdba6a028d4ccf1557a15a6ee0065e4f0dfadcde11096ccdd265bd5da'
for name, expected in EXPECTED.items():
    assert hashlib.sha256((ROOT/'Files'/name).read_bytes()).hexdigest() == expected, f'Frozen code not fully synced: {name}'
BUNDLE = ROOT/'Datasets/PHM15-diagnosis/heldout_test_v1'
assert hashlib.sha256((BUNDLE/'export_contract.json').read_bytes()).hexdigest() == EXPECTED_CONTRACT, 'Frozen export contract not fully synced.'
contract = json.loads((BUNDLE/'export_contract.json').read_text(encoding='utf-8'))
for name in ['test_evidence.jsonl','feature_dictionary.json','evaluation_protocol.json']:
    assert hashlib.sha256((BUNDLE/name).read_bytes()).hexdigest() == contract['files'][name], f'Frozen inputs not fully synced: {name}'
print('Frozen sample verified: 100 cases across ten held-out sites. No test labels opened.')


Mounted at /content/drive
Frozen sample verified: 100 cases across ten held-out sites. No test labels opened.


In [2]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime, then rerun all cells.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-literature-colab.txt', 'onnxruntime==1.31.1', 'lm-format-enforcer==0.11.3'], check=True)
print('Dependencies installed.')


GPU: NVIDIA L4
Dependencies installed.


In [3]:
for test in ['test_phm_two_stage.py', 'test_phm_heldout.py']:
    subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files', '-p', test, '-q'], check=True)
subprocess.run([sys.executable, 'Files/phm_two_stage.py', '--root', str(ROOT), '--max-cases', '100', '--heldout', '--prepare-only'], check=True)
print('Held-out preparation passed. Final cell executes or resumes up to 400 calls, then computes final metrics.')


Held-out preparation passed. Final cell executes or resumes up to 400 calls, then computes final metrics.


In [4]:
subprocess.run([sys.executable, 'Files/phm_two_stage.py', '--root', str(ROOT), '--max-cases', '100', '--heldout'], check=True)
latest = json.loads((ROOT/'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
run_folder = ROOT/latest['run_directory']
subprocess.run([sys.executable, 'Files/phm_final_metrics.py', '--root', str(ROOT), '--run', str(run_folder)], check=True)
print('Latest report:', latest)
print('Final metrics:', run_folder/'final_evaluation_metrics.json')
print('STOP: let Drive sync and send this report. Preserve all outcomes; do not tune or retry failed outputs after seeing test results.')


Latest report: {'run_id': '20261122T103151_paired_CD_two_stage_heldout_13ba4e1a', 'run_directory': 'Outputs/runs/20261122T103151_paired_CD_two_stage_heldout_13ba4e1a', 'status': 'completed_with_issues', 'updated_utc': '2026-11-22T14:29:59.590160+00:00'}
Final metrics: /content/drive/Othercomputers/My Laptop (1)/CP5403/Outputs/runs/20261122T103151_paired_CD_two_stage_heldout_13ba4e1a/final_evaluation_metrics.json
STOP: let Drive sync and send this report. Preserve all outcomes; do not tune or retry failed outputs after seeing test results.
